# ML MODEL-2 (NANDINI)

## loading dataset and importing libs

In [2]:
import pandas as pd
civil_df = pd.read_csv("civil_comments_sample.csv")
print(civil_df.shape)
civil_df.head()

(50000, 8)


,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,Socialists? What the hell does that have to do...,0.300000,0.000000,0.300000,0.0,0.200000,0.0,0.0
1,And I always thought that Eric Trump was the d...,0.808824,0.029412,0.088235,0.0,0.794118,0.0,0.0
2,I’m disappointed there’s no report from CB on ...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0
3,Nothing like subsidizing a country that is thr...,0.166667,0.000000,0.000000,0.0,0.166667,0.0,0.0
4,The only way to save the rhino is to legalize ...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0


In [3]:
print("Columns:")
print(civil_df.columns.tolist())

print("\nDataset information:")
civil_df.info()

print("\nMissing values:")
print(civil_df.isna().sum())

print("\nLabel score summaries:")
civil_df.describe()

Columns:
['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   text             50000 non-null  str    
 1   toxicity         50000 non-null  float64
 2   severe_toxicity  50000 non-null  float64
 3   obscene          50000 non-null  float64
 4   threat           50000 non-null  float64
 5   insult           50000 non-null  float64
 6   identity_attack  50000 non-null  float64
 7   sexual_explicit  50000 non-null  float64
dtypes: float64(7), str(1)
memory usage: 17.2 MB

Missing values:
text               0
toxicity           0
severe_toxicity    0
obscene            0
threat             0
insult             0
identity_attack    0
sexual_explicit    0
dtype: int64

Label score summaries:


,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,0.103082,0.004604,0.013997,0.009351,0.081054,0.022486,0.006496
std,0.196056,0.022900,0.065570,0.048949,0.174806,0.078174,0.044575
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.166667,0.000000,0.000000,0.000000,0.100000,0.000000,0.000000
max,1.000000,0.500000,1.000000,1.000000,1.000000,1.000000,0.909028


In [4]:
empty_or_symbols = (
    civil_df["text"].astype("string").str.strip().eq("")
    | civil_df["text"].astype("string").str.fullmatch(r"[\W_]*")
)

print("Duplicate rows:", civil_df.duplicated().sum())
print("Repeated comments:", civil_df["text"].duplicated().sum())
print("Empty or punctuation-only comments:", empty_or_symbols.sum())

Duplicate rows: 66
Repeated comments: 75
Empty or punctuation-only comments: 7


In [5]:
empty_or_symbols = (
    civil_df["text"].astype("string").str.strip().eq("")
    | civil_df["text"].astype("string").str.fullmatch(r"[\W_]*")
)

civil_df = civil_df.loc[~empty_or_symbols].copy()
civil_df = civil_df.drop_duplicates(subset="text", keep="first")
civil_df = civil_df.reset_index(drop=True)

print("Cleaned dataset shape:", civil_df.shape)
print("Remaining duplicate comments:", civil_df["text"].duplicated().sum())

Cleaned dataset shape: (49919, 8)
Remaining duplicate comments: 0


## Create Binary Toxicity Labels

The dataset scores show how many annotators agreed with each label. We will use 0.1 as the initial cutoff: scores at or above it become 1, and lower scores become 0. A comment is toxic if any toxicity category is positive.

In [6]:
label_columns = [
    "toxicity",
    "severe_toxicity",
    "obscene",
    "threat",
    "insult",
    "identity_attack",
    "sexual_explicit"
]

label_cutoff = 0.1

for column in label_columns:
    civil_df[column + "_label"] = (
        civil_df[column] >= label_cutoff
    ).astype(int)

binary_label_columns = [column + "_label" for column in label_columns]

civil_df["is_toxic"] = civil_df[binary_label_columns].max(axis=1)

print("Toxic and non-toxic comment counts:")
print(civil_df["is_toxic"].value_counts())

Toxic and non-toxic comment counts:
is_toxic
0    34378
1    15541
Name: count, dtype: int64


## Split the Data

We will set aside validation and test data so we can tune and evaluate the model on comments it has not trained on.

In [7]:
from sklearn.model_selection import train_test_split

train_df, remaining_df = train_test_split(
    civil_df,
    test_size=0.30,
    random_state=42,
    stratify=civil_df["is_toxic"]
)

validation_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=42,
    stratify=remaining_df["is_toxic"]
)

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

Training rows: 34943
Validation rows: 7488
Test rows: 7488


## Prepare the Comment Text

We will tokenize the comments, remove stop words while keeping negation words such as “not,” and lemmatize the remaining words. Stemming is another way to shorten words, but we’ll use lemmatization for this model instead of applying both.

In [8]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}
lemmatizer = WordNetLemmatizer()

def prepare_text(text):
    tokens = word_tokenize(str(text).lower())

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]

    return " ".join(tokens)

train_df["processed_text"] = train_df["text"].apply(prepare_text)
validation_df["processed_text"] = validation_df["text"].apply(prepare_text)
test_df["processed_text"] = test_df["text"].apply(prepare_text)

print(train_df["processed_text"].head())

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nandini\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nandini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Nandini\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


16932    cr meet el paso texas drink chinese restaurant...
1699     no need go back school think amendment contain...
34427    agree business regulated people california not...
17408    spare no cost even know going eat bill no not ...
8844     subsidy dairy farmer canada get consumer regre...
Name: processed_text, dtype: str


## Convert Text to TF-IDF Features

TF-IDF represents each comment as numbers. We will include single words and two-word phrases, and fit the vectorizer on training comments only.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000
)

tfidf_train = tfidf_vectorizer.fit_transform(train_df["processed_text"])
tfidf_validation = tfidf_vectorizer.transform(validation_df["processed_text"])
tfidf_test = tfidf_vectorizer.transform(test_df["processed_text"])

print("Training feature shape:", tfidf_train.shape)
print("Validation feature shape:", tfidf_validation.shape)
print("Test feature shape:", tfidf_test.shape)

Training feature shape: (34943, 30000)
Validation feature shape: (7488, 30000)
Test feature shape: (7488, 30000)


## Train the Toxicity Model

We will train one class-balanced Logistic Regression model to predict the seven toxicity categories. We can derive `is_toxic` from those category predictions.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

toxicity_model = MultiOutputClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

toxicity_model.fit(
    tfidf_train,
    train_df[binary_label_columns]
)

print("Toxicity model training complete.")

Toxicity model training complete.


## Set Category Cutoffs

We will choose a probability cutoff for each category using validation data. The cutoff with the best F1 score will balance catching toxic comments with avoiding false alarms.

In [12]:
from sklearn.metrics import precision_recall_curve

validation_probabilities = toxicity_model.predict_proba(tfidf_validation)
label_thresholds = {}

for i, label in enumerate(binary_label_columns):
    actual = validation_df[label].to_numpy()
    probabilities = validation_probabilities[i][:, 1]

    precision, recall, cutoffs = precision_recall_curve(actual, probabilities)
    f1_scores = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-10
    )

    label_thresholds[label] = cutoffs[f1_scores.argmax()]

print("Chosen cutoffs:")
for label, cutoff in label_thresholds.items():
    print(f"{label}: {cutoff:.3f}")

Chosen cutoffs:
toxicity_label: 0.469
severe_toxicity_label: 0.490
obscene_label: 0.605
threat_label: 0.659
insult_label: 0.498
identity_attack_label: 0.571
sexual_explicit_label: 0.589


This step chooses a probability cutoff for each category. The model is already trained; we’re deciding how confident it should be before calling a comment positive for a category.
For each category, the code:
1. Gets the model’s predicted probabilities for the validation comments.
2. Tries different cutoffs. For example, a cutoff of 0.30 marks a comment positive when the model’s probability is at least 30%.
3. Calculates an F1 score at each cutoff. F1 balances precision (how often positive predictions are correct) and recall (how many actual positive comments are caught).
4. Keeps the cutoff with the best F1 score.
The cutoffs are selected using validation data. We’ll save the separate test data for the final evaluation.

## Evaluate on the Test Data

We will use the cutoffs chosen from validation data to evaluate the model on the untouched test data. The table shows how well it detects each category and toxicity overall.

In [15]:
from sklearn.metrics import precision_recall_fscore_support

test_probabilities = toxicity_model.predict_proba(tfidf_test)

test_predictions = pd.DataFrame({
    label: (
        test_probabilities[i][:, 1] >= label_thresholds[label]
    ).astype(int)
    for i, label in enumerate(binary_label_columns)
}, index=test_df.index)

test_predictions["is_toxic"] = test_predictions[binary_label_columns].max(axis=1)

results = []

for label in binary_label_columns + ["is_toxic"]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_df[label],
        test_predictions[label],
        average="binary",
        zero_division=0
    )

    results.append({
        "Category": label,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Positive comments": int(test_df[label].sum())
    })

results_df = pd.DataFrame(results).set_index("Category").round(3)
results_df

,Precision,Recall,F1-score,Positive comments
Category,,,,
toxicity_label,0.545,0.692,0.610,2248
severe_toxicity_label,0.081,0.232,0.120,190
obscene_label,0.427,0.370,0.396,476
threat_label,0.310,0.312,0.311,349
insult_label,0.525,0.642,0.578,1858
identity_attack_label,0.424,0.522,0.468,758
sexual_explicit_label,0.403,0.380,0.391,213
is_toxic,0.532,0.724,0.613,2331


## Save the Toxicity Model

We will save the trained model, TF-IDF vectorizer, and chosen category cutoffs together. The saved file is for our project prototype; the test results show that some categories still need improvement.

In [16]:
import joblib

toxicity_bundle = {
    "model": toxicity_model,
    "vectorizer": tfidf_vectorizer,
    "thresholds": label_thresholds,
    "label_columns": binary_label_columns,
    "label_cutoff": label_cutoff
}

joblib.dump(toxicity_bundle, "toxicity_model.joblib")

print("Toxicity model saved.")

Toxicity model saved.
